In [22]:
import logging
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import pandas as pd

In [13]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [18]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [ ]:
def instrumentLookup(instrument_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instrument_df[instrument_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1
instrumentLookup(instruments_df,"ACC")


5633

In [ ]:
def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data
fetchOHLC("ACC","5minute",5)

,open,high,low,close,volume
date,,,,,
2025-04-01 09:15:00+05:30,1943.95,1957.90,1937.00,1941.50,8132
2025-04-01 09:20:00+05:30,1939.20,1952.35,1939.20,1948.00,4259
2025-04-01 09:25:00+05:30,1948.00,1957.35,1947.50,1956.20,7965
2025-04-01 09:30:00+05:30,1954.45,1960.85,1954.45,1958.95,5268
2025-04-01 09:35:00+05:30,1958.90,1958.90,1954.35,1955.75,1643
...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1963.95,1969.10,1963.95,1969.10,2213
2025-04-04 15:10:00+05:30,1969.10,1969.10,1965.00,1967.90,1376
2025-04-04 15:15:00+05:30,1967.90,1968.90,1965.30,1967.55,1870


In [27]:
import pandas as pd
import datetime as dt

def fetchOHLCExtended(ticker, inception_date, interval):
    """Extracts historical data and outputs in the form of a DataFrame
       inception date string format - dd-mm-yyyy"""
    instrument = instrumentLookup(instruments_df, ticker)
    from_date = dt.datetime.strptime(inception_date, '%d-%m-%Y')
    to_date = dt.date.today()
    data = pd.DataFrame(columns=['date', 'open', 'high', 'low', 'close', 'volume'])

    while True:
        if from_date.date() >= (dt.date.today() - dt.timedelta(100)):
            historical_data = pd.DataFrame(kite.historical_data(instrument, from_date, dt.date.today(), interval))
            data = pd.concat([data, historical_data], ignore_index=True)
            break
        else:
            to_date = from_date + dt.timedelta(100)
            historical_data = pd.DataFrame(kite.historical_data(instrument, from_date, to_date, interval))
            data = pd.concat([data, historical_data], ignore_index=True)
            from_date = to_date

    data.set_index("date", inplace=True)
    return data

# Example usage
fetchOHLCExtended("INFY", "01-01-2024", "5minute")


,open,high,low,close,volume
date,,,,,
2024-01-01 09:15:00+05:30,1539.00,1542.90,1535.25,1538.25,76413
2024-01-01 09:20:00+05:30,1538.15,1539.50,1537.10,1538.60,36570
2024-01-01 09:25:00+05:30,1538.65,1542.55,1538.55,1540.60,33452
2024-01-01 09:30:00+05:30,1541.00,1541.20,1539.50,1541.05,44560
2024-01-01 09:35:00+05:30,1540.75,1542.00,1540.00,1540.00,16008
...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1450.40,1453.55,1449.00,1453.50,248967
2025-04-04 15:10:00+05:30,1453.10,1453.10,1450.60,1451.20,212260
2025-04-04 15:15:00+05:30,1451.25,1453.65,1450.25,1452.45,268750
